# 代码文件3：Task2指标数据分析

本notebook用于分析task2的指标数据，创建task2指标数据表单和论文需要的图片。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# 表格美化显示
try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False
    print("提示: 安装 tabulate 可以获得更好的表格显示效果: pip install tabulate")

try:
    from rich.console import Console
    from rich.table import Table
    from rich import box
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    print("提示: 安装 rich 可以获得彩色表格显示效果: pip install rich")

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表美化样式
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'seaborn-darkgrid')
sns.set_palette("husl")  # 使用更鲜艳的颜色调色板
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# 优化pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)  # 不限制列宽，让内容完整显示
pd.set_option('display.precision', 3)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if pd.notna(x) else '')
pd.set_option('display.expand_frame_repr', False)  # 不换行显示DataFrame

# 定义美化表格显示函数
def display_table(df, title="", max_rows=20, use_rich=True):
    """美化显示DataFrame表格"""
    if title:
        print(f"\n{'='*80}")
        print(f"{title:^80}")
        print(f"{'='*80}")
    
    # 优先使用rich显示（如果可用且数据不太大）
    if HAS_RICH and use_rich and len(df) <= 50 and len(df.columns) <= 15:
        console = Console(width=None)  # 不限制控制台宽度
        table = Table(show_header=True, header_style="bold magenta", box=box.ROUNDED, expand=True, 
                     show_lines=False, pad_edge=False)
        
        # 添加列（不换行，允许更宽的列，设置最小宽度确保列名完整显示）
        for col in df.columns:
            col_str = str(col)
            # 设置最小宽度为列名长度，确保列名完整显示
            min_width = max(len(col_str), 10)
            table.add_column(col_str, style="cyan", no_wrap=True, min_width=min_width, overflow="fold")
        
        # 添加行
        for idx, row in df.head(max_rows).iterrows():
            # 格式化数值，保留3位小数
            formatted_row = []
            for val in row:
                if pd.notna(val):
                    if isinstance(val, (int, float)):
                        formatted_row.append(f"{val:.3f}")
                    else:
                        formatted_row.append(str(val))
                else:
                    formatted_row.append('N/A')
            table.add_row(*formatted_row)
        
        console.print(table)
        if len(df) > max_rows:
            print(f"\n... (还有 {len(df) - max_rows} 行未显示)")
    
    # 使用tabulate显示
    elif HAS_TABULATE:
        # 临时设置pandas选项确保列内容完整
        old_max_colwidth = pd.get_option('display.max_colwidth')
        pd.set_option('display.max_colwidth', None)
        
        try:
            # 格式化DataFrame，确保数值显示为3位小数
            display_df = df.copy()
            for col in display_df.select_dtypes(include=[np.number]).columns:
                display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
            
            if len(display_df) > max_rows:
                print(f"\n显示前{max_rows}行（共{len(display_df)}行）:")
                print(tabulate(display_df.head(max_rows), headers='keys', tablefmt='grid', 
                              showindex=False, stralign='left', numalign='right'))
                print(f"\n... (还有 {len(display_df) - max_rows} 行)")
            else:
                print(tabulate(display_df, headers='keys', tablefmt='grid', 
                              showindex=False, stralign='left', numalign='right'))
        finally:
            pd.set_option('display.max_colwidth', old_max_colwidth)
    
    # 回退到pandas默认显示
    else:
        if len(df) > max_rows:
            print(f"\n显示前{max_rows}行（共{len(df)}行）:")
            print(df.head(max_rows).to_string(max_colwidth=None))
            print(f"\n... (还有 {len(df) - max_rows} 行)")
        else:
            print(df.to_string(max_colwidth=None))
    
    print(f"\n形状: {df.shape[0]} 行 × {df.shape[1]} 列")

# 数据根目录
# Resolve VCG-Bench root from the current notebook working directory.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    return start

BASE_DIR = find_repo_root()
DATA_DIR = BASE_DIR / 'data'
ANALYSIS_DIR = DATA_DIR / 'analysis'

# 创建必要的目录结构
(ANALYSIS_DIR / 'task2_analysis' / 'tables').mkdir(parents=True, exist_ok=True)
(ANALYSIS_DIR / 'task2_analysis' / 'figures').mkdir(parents=True, exist_ok=True)

print(f"数据根目录: {BASE_DIR}")
print(f"分析结果目录: {ANALYSIS_DIR}")


In [ ]:
# 读取Task2详细结果和难度分层数据

# 读取detailed_results.json
task2_results_path = DATA_DIR / 'task2_evaluation' / 'detailed_results.json'
with open(task2_results_path, 'r', encoding='utf-8') as f:
    task2_data = json.load(f)

# 检查JSON结构：可能是字典（包含samples或instructions键）或直接是列表
if isinstance(task2_data, dict):
    if 'samples' in task2_data:
        task2_results = task2_data['samples']
    elif 'instructions' in task2_data:
        task2_results = task2_data['instructions']
    else:
        task2_results = []
elif isinstance(task2_data, list):
    task2_results = task2_data
else:
    task2_results = []

print(f"Task2结果记录数: {len(task2_results)}")

# 读取难度分层数据
difficulty_levels_path = ANALYSIS_DIR / 'difficulty_stratification' / 'tables' / 'image_difficulty_levels.csv'
difficulty_levels_df = pd.read_csv(difficulty_levels_path)

print(f"难度分层数据记录数: {len(difficulty_levels_df)}")
display_table(difficulty_levels_df.head(5), title="难度分层数据前5条", max_rows=5)


In [ ]:
# 数据清洗和合并
# 将JSON数据转换为DataFrame，并合并难度信息

def extract_metric_score(metric_data):
    """提取指标的score值，如果success=false或值为None则返回None"""
    if metric_data is None:
        return None
    if isinstance(metric_data, dict):
        if metric_data.get('success', True) == False:
            return None
        return metric_data.get('score', None)
    return metric_data

def extract_instruction_level_from_id(instruction_id):
    """从instruction_id中提取指令难度
    命名规则：inst_{难度}_{编号}，如 inst_easy_001 -> Easy
    """
    if not instruction_id or pd.isna(instruction_id):
        return None
    
    instruction_id_str = str(instruction_id)
    # 匹配模式：inst_{难度}_{编号}
    # 提取难度部分（inst_ 和第一个 _ 之间的部分）
    if instruction_id_str.startswith('inst_'):
        parts = instruction_id_str.split('_')
        if len(parts) >= 2:
            level_str = parts[1].lower()
            # 映射到标准格式：Easy, Medium, Hard
            level_mapping = {
                'easy': 'Easy',
                'medium': 'Medium',
                'hard': 'Hard'
            }
            return level_mapping.get(level_str, None)
    return None

# 展开JSON数据为DataFrame
rows = []
for item in task2_results:
    instruction_id = item.get('instruction_id')
    instruction_level = item.get('instruction_level')
    
    # 优先从instruction_id中提取指令难度（因为命名规则很明确：inst_easy_001）
    # 如果提取失败，再使用原有的instruction_level
    extracted_level = extract_instruction_level_from_id(instruction_id)
    if extracted_level:
        instruction_level = extracted_level
    # 如果instruction_level为None、'unknown'或其他无效值，且无法从instruction_id提取，则保持None
    elif not instruction_level or str(instruction_level).lower() in ['unknown', 'none', '']:
        instruction_level = None
    
    row = {
        'sample_id': item.get('sample_id'),
        'domain': item.get('domain'),
        'model': item.get('model'),
        'instruction_id': instruction_id,
        'instruction_level': instruction_level,
    }
    
    # 提取各指标的score值
    metrics = item.get('metrics', {})
    row['modified_xml_execution_success_rate'] = extract_metric_score(metrics.get('modified_xml_execution_success_rate'))
    row['modified_xml_token_count'] = extract_metric_score(metrics.get('modified_xml_token_count'))
    row['modification_json_token_count'] = extract_metric_score(metrics.get('modification_json_token_count'))
    row['style_consistency_score_task2'] = extract_metric_score(metrics.get('style_consistency_score_task2'))
    row['xdrfr'] = extract_metric_score(metrics.get('xdrfr'))
    row['xml_edit_distance'] = extract_metric_score(metrics.get('xml_edit_distance'))
    
    rows.append(row)

task2_df = pd.DataFrame(rows)

# 检查DataFrame是否为空
if len(task2_df) == 0:
    print("警告: Task2结果数据为空，无法进行后续分析")
    print("请检查 detailed_results.json 文件是否正确加载")
else:
    print(f"成功创建DataFrame，共 {len(task2_df)} 条记录")
    print(f"列名: {list(task2_df.columns)}")
    
    # 清洗：删除modification_json_token_count为0的样本（如果列存在）
    if 'modification_json_token_count' in task2_df.columns:
        task2_df = task2_df[task2_df['modification_json_token_count'] != 0].copy()
        print(f"清洗后剩余 {len(task2_df)} 条记录")
    
    # 显示instruction_level分布（清洗后）
    print(f"\ninstruction_level分布（清洗后）:")
    inst_level_counts = task2_df['instruction_level'].value_counts()
    display_table(pd.DataFrame({
        'instruction_level': inst_level_counts.index,
        '记录数': inst_level_counts.values
    }), title="instruction_level分布", max_rows=10)
    
    # 检查是否有无法提取指令难度的记录
    missing_inst_level = task2_df[task2_df['instruction_level'].isna()]
    if len(missing_inst_level) > 0:
        print(f"\n⚠️ 警告: 有 {len(missing_inst_level)} 条记录无法提取instruction_level")
        print("这些记录的instruction_id示例:")
        sample_ids = missing_inst_level['instruction_id'].dropna().head(10).tolist()
        for inst_id in sample_ids:
            print(f"  - {inst_id}")
    else:
        print("\n✓ 所有记录的instruction_level都已成功提取")
    
    # 合并难度信息
    task2_df = task2_df.merge(
        difficulty_levels_df[['sample_id', 'domain', 'difficulty_level']],
        on=['sample_id', 'domain'],
        how='left'
    )
    
    # 保存合并后的数据
    output_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_merged_data.csv'
    task2_df.to_csv(output_path, index=False, encoding='utf-8')
    
    print(f"合并后数据记录数: {len(task2_df)}")
    print(f"列数: {len(task2_df.columns)}")
    print(f"\n列名: {list(task2_df.columns)}")
    
    # 显示缺失值统计
    missing_stats = pd.DataFrame({
        '缺失数量': task2_df.isnull().sum(),
        '缺失比例(%)': (task2_df.isnull().sum() / len(task2_df) * 100).round(2)
    })
    missing_stats = missing_stats[missing_stats['缺失数量'] > 0].sort_values('缺失数量', ascending=False)
    if len(missing_stats) > 0:
        display_table(missing_stats, title="缺失值统计")
    else:
        print("\n✓ 无缺失值")
    
    # 显示前5条数据
    display_table(task2_df.head(5), title="前5条数据预览")
    print(f"\n已保存到: {output_path}")


In [ ]:
# 按模型分组统计
# 对每个模型的指标计算统计值（排除None/Null/success=false的样本）

def safe_statistics(series, stat_name):
    """安全计算统计量，排除None/Null值"""
    valid_series = series.dropna()
    if len(valid_series) == 0:
        return None
    
    if stat_name == 'mean':
        return valid_series.mean()
    elif stat_name == 'median':
        return valid_series.median()
    elif stat_name == 'std':
        return valid_series.std()
    elif stat_name == 'min':
        return valid_series.min()
    elif stat_name == 'max':
        return valid_series.max()
    elif stat_name == 'count':
        return len(valid_series)
    return None

# 定义要统计的指标
metrics_list = ['modified_xml_execution_success_rate', 'modified_xml_token_count',
                'modification_json_token_count', 'style_consistency_score_task2',
                'xdrfr', 'xml_edit_distance']
statistics_list = ['mean', 'median', 'std', 'min', 'max', 'count']

# 按模型分组统计
model_stats = []
for model in task2_df['model'].unique():
    model_data = task2_df[task2_df['model'] == model]
    
    row = {'model': model}
    
    for metric in metrics_list:
        metric_series = model_data[metric]
        for stat in statistics_list:
            col_name = f'{metric}_{stat}'
            row[col_name] = safe_statistics(metric_series, stat)
    
    row['valid_samples_count'] = len(model_data)
    model_stats.append(row)

task2_by_model_df = pd.DataFrame(model_stats)

# 保存统计表格
output_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_by_model.csv'
task2_by_model_df.to_csv(output_path, index=False, encoding='utf-8')

# 选择均值列进行显示，并简化列名
mean_cols = [col for col in task2_by_model_df.columns if col.endswith('_mean')]
display_df = task2_by_model_df[['model'] + mean_cols].copy()

# 简化列名：去掉 _mean 后缀，并简化指标名称
column_mapping = {
    'modified_xml_execution_success_rate_mean': 'Exec Success Rate',
    'modified_xml_token_count_mean': 'XML Tokens',
    'modification_json_token_count_mean': 'JSON Tokens',
    'style_consistency_score_task2_mean': 'SCS',
    'xdrfr_mean': 'XDRFR',
    'xml_edit_distance_mean': 'Edit Distance'
}
display_df = display_df.rename(columns=column_mapping)

display_table(display_df, title="Task2按模型分组统计（均值）", max_rows=50)
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制模型对比图（图1）
# 使用热力图展示不同模型的性能对比

# 选择主要指标
main_metrics = ['modified_xml_execution_success_rate_mean', 'xdrfr_mean',
                'style_consistency_score_task2_mean', 'xml_edit_distance_mean']

# 构建矩阵：行=模型，列=指标
heatmap_data = task2_by_model_df.set_index('model')[main_metrics]

# 重命名列以便显示
heatmap_data.columns = ['Exec Success', 'XDRFR', 'SCS', 'Edit Distance']

# 绘制热力图
plt.figure(figsize=(12, max(8, len(heatmap_data) * 0.5)))
sns.heatmap(heatmap_data, annot=True, fmt='.5f', cmap='YlOrRd',
            cbar_kws={'label': 'Score'}, linewidths=0.5, linecolor='gray')
plt.title('Task2 Model Performance Comparison (Heatmap)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Metrics', fontsize=12)
plt.ylabel('Models', fontsize=12)
plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task2_analysis' / 'figures' / 'task2_model_comparison.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 按图片难度等级分组统计
# 计算每个模型在不同图片难度等级下的表现

difficulty_stats = []
for difficulty in task2_df['difficulty_level'].dropna().unique():
    for model in task2_df['model'].unique():
        model_difficulty_data = task2_df[
            (task2_df['model'] == model) &
            (task2_df['difficulty_level'] == difficulty)
        ]
        
        if len(model_difficulty_data) == 0:
            continue
        
        row = {
            'difficulty_level': difficulty,
            'model': model
        }
        
        for metric in metrics_list:
            metric_series = model_difficulty_data[metric]
            for stat in statistics_list:
                col_name = f'{metric}_{stat}'
                row[col_name] = safe_statistics(metric_series, stat)
        
        row['valid_samples_count'] = len(model_difficulty_data)
        difficulty_stats.append(row)

task2_by_difficulty_df = pd.DataFrame(difficulty_stats)

# 保存统计表格
output_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_by_difficulty.csv'
task2_by_difficulty_df.to_csv(output_path, index=False, encoding='utf-8')

# 选择主要列进行显示
display_cols = ['difficulty_level', 'model'] + [col for col in task2_by_difficulty_df.columns if col.endswith('_mean')]
display_table(task2_by_difficulty_df[display_cols], title="Task2按图片难度等级分组统计（主要指标均值）", max_rows=30)
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制按图片难度等级的性能表现图（图2）
# 使用分组柱状图展示模型在不同图片难度下的表现

# 选择主要指标和模型
main_metrics_plot = ['xdrfr_mean', 'style_consistency_score_task2_mean']
# 使用排序后的模型列表，确保包含所有重要模型（特别是Qwen 32B）
all_models = sorted(task2_by_difficulty_df['model'].unique())
top_models = all_models[:8]  # 选择前8个模型（按字母排序）
print(f"选择的模型: {top_models}")

fig, axes = plt.subplots(1, len(main_metrics_plot), figsize=(16, 6))

for idx, metric in enumerate(main_metrics_plot):
    ax = axes[idx]
    
    # 准备数据
    plot_data = task2_by_difficulty_df[
        task2_by_difficulty_df['model'].isin(top_models)
    ].pivot_table(
        index='difficulty_level',
        columns='model',
        values=metric
    )
    
    # 按难度等级排序：Easy -> Medium -> Hard
    difficulty_order = ['Easy', 'Medium', 'Hard']
    # 只保留存在的难度等级
    available_difficulties = [d for d in difficulty_order if d in plot_data.index]
    plot_data = plot_data.reindex(available_difficulties)
    
    # 绘制分组柱状图
    plot_data.plot(kind='bar', ax=ax, width=0.8, alpha=0.8)
    
    metric_name = metric.replace('_mean', '').replace('_', ' ').title()
    ax.set_title(f'{metric_name} by Image Difficulty Level', fontsize=12, fontweight='bold')
    ax.set_xlabel('Difficulty Level', fontsize=11)
    ax.set_ylabel('Score', fontsize=11)
    ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task2_analysis' / 'figures' / 'task2_by_difficulty.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 读取并处理按指令难度数据
# 直接从已清洗的task2_df生成统计，确保instruction_level正确提取

# 使用已清洗的task2_df数据，按指令难度分组统计
difficulty_stats_inst = []
for instruction_level in task2_df['instruction_level'].dropna().unique():
    for model in task2_df['model'].unique():
        model_inst_data = task2_df[
            (task2_df['model'] == model) &
            (task2_df['instruction_level'] == instruction_level)
        ]
        
        if len(model_inst_data) == 0:
            continue
        
        row = {
            'instruction_level': instruction_level,
            'model': model
        }
        
        for metric in metrics_list:
            metric_series = model_inst_data[metric]
            for stat in statistics_list:
                col_name = f'{metric}_{stat}'
                row[col_name] = safe_statistics(metric_series, stat)
        
        row['valid_samples_count'] = len(model_inst_data)
        difficulty_stats_inst.append(row)

task2_instruction_df = pd.DataFrame(difficulty_stats_inst)

# 数据验证和清洗
print(f"按指令难度数据记录数: {len(task2_instruction_df)}")
print(f"列名: {list(task2_instruction_df.columns)}")

# 显示前10条数据
display_table(task2_instruction_df[['instruction_level', 'model'] + [col for col in task2_instruction_df.columns if col.endswith('_mean')]].head(10), 
              title="Task2按指令难度数据（前10条）", max_rows=10)

print(f"\n指令难度分布:")
if 'instruction_level' in task2_instruction_df.columns:
    inst_counts = task2_instruction_df['instruction_level'].value_counts()
    display_table(pd.DataFrame({
        '指令难度': inst_counts.index,
        '记录数': inst_counts.values
    }), title="指令难度分布", max_rows=10)
    
    # 检查是否有unknown
    if 'unknown' in inst_counts.index or 'Unknown' in inst_counts.index:
        print("\n⚠️ 警告: 发现 'unknown' 指令难度，正在尝试从instruction_id中提取...")
        # 如果有unknown，尝试从task2_df中重新提取
        unknown_mask = task2_df['instruction_level'].isin(['unknown', 'Unknown', None])
        if unknown_mask.sum() > 0:
            print(f"发现 {unknown_mask.sum()} 条记录的instruction_level为unknown或None")
            # 重新提取
            for idx in task2_df[unknown_mask].index:
                instruction_id = task2_df.loc[idx, 'instruction_id']
                extracted_level = extract_instruction_level_from_id(instruction_id)
                if extracted_level:
                    task2_df.loc[idx, 'instruction_level'] = extracted_level
                    print(f"  从 {instruction_id} 提取到 {extracted_level}")
            
            # 重新生成统计
            difficulty_stats_inst = []
            for instruction_level in task2_df['instruction_level'].dropna().unique():
                for model in task2_df['model'].unique():
                    model_inst_data = task2_df[
                        (task2_df['model'] == model) &
                        (task2_df['instruction_level'] == instruction_level)
                    ]
                    
                    if len(model_inst_data) == 0:
                        continue
                    
                    row = {
                        'instruction_level': instruction_level,
                        'model': model
                    }
                    
                    for metric in metrics_list:
                        metric_series = model_inst_data[metric]
                        for stat in statistics_list:
                            col_name = f'{metric}_{stat}'
                            row[col_name] = safe_statistics(metric_series, stat)
                    
                    row['valid_samples_count'] = len(model_inst_data)
                    difficulty_stats_inst.append(row)
            
            task2_instruction_df = pd.DataFrame(difficulty_stats_inst)
            print(f"\n重新生成后，按指令难度数据记录数: {len(task2_instruction_df)}")
            
            # 再次显示指令难度分布
            inst_counts = task2_instruction_df['instruction_level'].value_counts()
            display_table(pd.DataFrame({
                '指令难度': inst_counts.index,
                '记录数': inst_counts.values
            }), title="指令难度分布（清洗后）", max_rows=10)

# 保存（如果需要处理）
output_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_by_instruction_difficulty.csv'
task2_instruction_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制按指令难度的性能表现图（图3）
# 展示模型在不同指令难度下的表现（Task2特有）

# 选择主要指标
main_metrics_inst = ['xdrfr_mean', 'style_consistency_score_task2_mean']

# 检查数据格式，可能需要处理多级索引
if 'instruction_level' in task2_instruction_df.columns and 'model' in task2_instruction_df.columns:
    # 使用排序后的模型列表，确保包含所有重要模型（特别是Qwen 32B）
    all_models_inst = sorted(task2_instruction_df['model'].unique())
    top_models_inst = all_models_inst[:8]  # 选择前8个模型（按字母排序）
    print(f"选择的模型: {top_models_inst}")
    
    fig, axes = plt.subplots(1, len(main_metrics_inst), figsize=(16, 6))
    
    for idx, metric in enumerate(main_metrics_inst):
        if metric not in task2_instruction_df.columns:
            continue
            
        ax = axes[idx]
        
        # 准备数据
        plot_data = task2_instruction_df[
            task2_instruction_df['model'].isin(top_models_inst)
        ].pivot_table(
            index='instruction_level',
            columns='model',
            values=metric,
            aggfunc='mean'
        )
        
        # 绘制分组柱状图
        plot_data.plot(kind='bar', ax=ax, width=0.8, alpha=0.8)
        
        metric_name = metric.replace('_mean', '').replace('_', ' ').title()
        ax.set_title(f'{metric_name} by Instruction Difficulty', fontsize=12, fontweight='bold')
        ax.set_xlabel('Instruction Level', fontsize=11)
        ax.set_ylabel('Score', fontsize=11)
        ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    
    # 保存图片
    output_path = ANALYSIS_DIR / 'task2_analysis' / 'figures' / 'task2_by_instruction_difficulty.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"已保存图片到: {output_path}")
    
    plt.show()
else:
    print("数据格式不符合预期，请检查CSV文件结构")


In [ ]:
# 读取并处理SCS按维度数据

scs_path = DATA_DIR / 'task2_evaluation' / 'all_models_scs_by_dimension.csv'
task2_scs_df = pd.read_csv(scs_path)

# 数据验证和清洗
print(f"SCS数据记录数: {len(task2_scs_df)}")

display_table(task2_scs_df, title="SCS按维度数据", max_rows=30)

print(f"\n维度分布:")
dim_counts = task2_scs_df['dimension'].value_counts()
display_table(pd.DataFrame({
    '维度': dim_counts.index,
    '记录数': dim_counts.values
}), title="维度分布", max_rows=10)

# 保存（如果需要处理）
output_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_scs_by_dimension.csv'
task2_scs_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制SCS按维度表现图（图4）
# 展示模型在不同SCS维度上的表现

# 选择主要模型（使用排序后的列表，确保包含所有重要模型）
all_models_scs = sorted(task2_scs_df['model'].unique())
top_models_scs = all_models_scs[:8]  # 选择前8个模型（按字母排序）
print(f"选择的模型: {top_models_scs}")

# Task2的SCS维度通常是aesthetic_quality和style_consistency
dimensions = task2_scs_df['dimension'].unique()

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(dimensions))
width = 0.12

for i, model in enumerate(top_models_scs):
    model_data = task2_scs_df[task2_scs_df['model'] == model]
    scores = []
    stds = []
    for dim in dimensions:
        dim_data = model_data[model_data['dimension'] == dim]
        score = dim_data['mean_score'].values[0] if len(dim_data) > 0 else 0
        std = dim_data['std_score'].values[0] if len(dim_data) > 0 else 0
        scores.append(score)
        stds.append(std)
    
    model_short = model.split('/')[-1] if '/' in model else model
    bars = ax.bar(x + i*width, scores, width, label=model_short, alpha=0.8)
    # 添加误差棒
    ax.errorbar(x + i*width, scores, yerr=stds, fmt='none', color='black', capsize=3, linewidth=1)

ax.set_xlabel('SCS Dimension', fontsize=12)
ax.set_ylabel('Mean Score', fontsize=12)
ax.set_title('Style Consistency Score by Dimension (Task2)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (len(top_models_scs) - 1) / 2)
ax.set_xticklabels([d.replace('_', ' ').title() for d in dimensions], fontsize=10)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_ylim([0, 10])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task2_analysis' / 'figures' / 'task2_scs_by_dimension.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制XDRFR按模型和图片难度等级的分组柱状图
# X轴是每个模型，Y轴是XDRFR，每个模型有3个柱子（Easy、Medium、Hard）

# 难度等级顺序
difficulty_order = ['Easy', 'Medium', 'Hard']

# 获取所有模型
all_models = sorted(task2_by_difficulty_df['model'].unique())
num_models = len(all_models)
num_difficulties = len(difficulty_order)

# 创建图表
fig, ax = plt.subplots(figsize=(16, 6))

# 计算合适的bar宽度：确保每组bar不会重叠
max_group_width = 0.85  # 每组bar的最大总宽度
width = max_group_width / num_difficulties  # 根据难度等级数量动态调整宽度

# 准备数据
x = np.arange(num_models)

# 为每个难度等级绘制bar
colors = plt.cm.Pastel1(np.linspace(0, 1, num_difficulties))  # 使用不同的颜色
for i, difficulty in enumerate(difficulty_order):
    scores = []
    for model in all_models:
        model_difficulty_data = task2_by_difficulty_df[
            (task2_by_difficulty_df['model'] == model) & 
            (task2_by_difficulty_df['difficulty_level'] == difficulty)
        ]
        score = model_difficulty_data['xdrfr_mean'].values[0] if len(model_difficulty_data) > 0 else 0
        scores.append(score)
    
    # 计算每组内的偏移量，使整个组居中
    offset_in_group = (i - (num_difficulties - 1) / 2) * width
    ax.bar(x + offset_in_group, scores, width, label=difficulty, alpha=0.8, color=colors[i])

# 设置x轴标签（模型名称的简短形式）
model_labels = [model.split('/')[-1] if '/' in model else model for model in all_models]
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Task 2 XDRFR', fontsize=12, fontweight='bold')
ax.set_title('XDRFR by Model and Image Difficulty Level', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=9)
ax.legend(title='Difficulty Level', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.set_ylim([0, 1.05])  # XDRFR是0-1的满足率
ax.grid(True, alpha=0.3, axis='y')
# 设置x轴范围，确保有足够的空间显示所有bar
ax.set_xlim([-0.5, num_models - 0.5])

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task2_analysis' / 'figures' / 'task2_xdrfr_by_difficulty.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

# 打印数据摘要
print("\n数据摘要:")
for model in all_models:
    model_short = model.split('/')[-1] if '/' in model else model
    scores = []
    for difficulty in difficulty_order:
        model_difficulty_data = task2_by_difficulty_df[
            (task2_by_difficulty_df['model'] == model) & 
            (task2_by_difficulty_df['difficulty_level'] == difficulty)
        ]
        score = model_difficulty_data['xdrfr_mean'].values[0] if len(model_difficulty_data) > 0 else None
        scores.append(score)
    
    easy_str = f"{scores[0]:.3f}" if scores[0] is not None else "N/A"
    medium_str = f"{scores[1]:.3f}" if scores[1] is not None else "N/A"
    hard_str = f"{scores[2]:.3f}" if scores[2] is not None else "N/A"
    print(f"{model_short}: Easy={easy_str}, Medium={medium_str}, Hard={hard_str}")

plt.show()

In [ ]:
# 指标说明文档

print("="*80)
print("Task2指标说明")
print("="*80)

print("\n1. Style Consistency Score Task2 - Task2风格一致性分数")
print("   评分范围: 0-10分")
print("   评分维度:")
print("     - aesthetic_quality (美学质量): 评估整体视觉效果的美观程度")
print("     - style_consistency (风格一致性): 评估修改后的XML代码在保持原有风格方面的表现")
print("   说明: 评估修改后的XML代码在保持原有风格方面的表现")

print("\n2. XDRFR (XML-based Diagram Requirement Fulfillment Rate)")
print("   评分范围: 0-1（满足率）")
print("   说明: 基于XML的图表需求满足率，评估模型生成的修改是否满足指令要求")
print("   评估方式: 通过一系列关于修改的问答，判断是否满足指令要求")
print("   例如: 是否删除了指定节点、是否修改了指定属性等")

print("\n3. Modified XML Execution Success Rate - 修改后XML执行成功率")
print("   评分范围: 0-1（成功率）")
print("   说明: 评估修改后的XML代码能否成功渲染")

print("\n4. Modified XML Token Count - 修改后XML token数")
print("   说明: 修改后的XML代码的token数量")

print("\n5. Modification JSON Token Count - 修改指令JSON token数")
print("   说明: 修改指令JSON的token数量，反映指令复杂度")

print("\n6. XML Edit Distance - XML编辑距离")
print("   评分范围: 0-1（归一化距离，越小表示差异越小）")
print("   说明: 评估修改后的XML与原始XML的差异程度")
print("="*80)
